# Task 1 (6p)
Your task is to modify the custom implementation of MultiHeadAttention. This custom implementation, currently, enables each token to attent to every other token.


Your job is to change this behavior in a specific way.
Let $S$ be our input sequence of length $2 \cdot k$:
- tokens on positions $i \lt k$ should attend to prefix of $S$ of length $k$ ($S[:k]$) - every token up to position k
- tokens on positions $i \ge k$ should attend to prefix of $S$  of length $i + 1$ ($S[:i + 1]$) - every previous token and itself

(Note: You can assume the sequence length is always an even number).

In [1]:
import torch
import math
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_head):
      super().__init__()
      self.d_model = d_model
      self.num_heads = num_heads
      self.d_head = d_head

      self.W_Q = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_K = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_V = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_O = torch.nn.Linear(num_heads*d_head, d_model, bias=True)

    def forward(self, x):

      seq_len, batch_size, _ = x.shape

      Q = self.W_Q(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      K = self.W_K(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      V = self.W_V(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)

      scaled_QK = torch.einsum("ibhd,jbhd->bhij", Q, K) / math.sqrt(self.d_head)
      # shape of scaled_QK is (batch_size, num_heads, seq_len, seq_len)

      #TODO
      k = seq_len // 2
      row_indices = torch.arange(seq_len).unsqueeze(1)
      col_indices = torch.arange(seq_len).unsqueeze(0)

      mask = torch.full((seq_len, seq_len), -torch.inf)
      condition = (((row_indices >= k) & (row_indices >= col_indices)) | ((row_indices < k) & (col_indices < k)))
      mask[condition] = 0

      scaled_QK = scaled_QK + mask.view(1, 1, seq_len, seq_len)                                          

      #ENDTODO

      weights = F.softmax(scaled_QK, -1)
      attention = torch.einsum("bhij,jbhd->ibhd", weights, V)

      result = self.W_O(attention.reshape(seq_len, batch_size,self.num_heads * self.d_head))

      return result, weights

In [2]:
# Test your solution
d_model = 10
num_heads= 4
d_head = 5
k = 10
batch_size = 16

mha = MultiHeadAttention(d_model, num_heads, d_head)
batched_x= torch.randn((2*k, batch_size, d_model))
with torch.no_grad():
  result, weights = mha(batched_x)
print("Result:", result)
print("Weights:", weights)

Result: tensor([[[-0.1946, -0.1997, -0.3567,  ...,  0.1251,  0.5307, -0.0613],
         [-0.1744, -0.1261, -0.0971,  ..., -0.1401,  0.5201, -0.3389],
         [-0.1597, -0.2635, -0.1865,  ...,  0.0581,  0.5028, -0.2176],
         ...,
         [-0.1757, -0.1298, -0.3366,  ...,  0.1198,  0.5613, -0.1872],
         [-0.3801, -0.0668, -0.0305,  ...,  0.1237,  0.4009, -0.3069],
         [-0.2134, -0.2301, -0.2088,  ...,  0.1469,  0.4347, -0.2019]],

        [[-0.1999, -0.1982, -0.3016,  ...,  0.0825,  0.4733, -0.0944],
         [-0.2113, -0.1731, -0.2110,  ..., -0.0118,  0.4779, -0.2819],
         [-0.1119, -0.3329, -0.1473,  ...,  0.1163,  0.4222, -0.2508],
         ...,
         [-0.1898, -0.1784, -0.2889,  ...,  0.1370,  0.5664, -0.2301],
         [-0.3209, -0.0510, -0.0855,  ...,  0.0406,  0.4942, -0.2930],
         [-0.2407, -0.1424, -0.2054,  ...,  0.2245,  0.5347, -0.2810]],

        [[-0.1430, -0.1738, -0.2801,  ...,  0.0401,  0.5207, -0.0914],
         [-0.2050, -0.0607, -0.1365, 